In [ ]:
!pip install -q -U keras-hub==0.21.1
!pip install  -q -U keras

In [ ]:
import pandas as pd
import pyarrow as pa
import keras
import keras_hub

**Format Sinhala Dataset.**
<br>
This model we trained a dataset that includes prompts and responces.
[Dataset](https://huggingface.co/datasets/Thimira/sinhala-llm-dataset-llama-prompt-format)


Downlaod the Dataset in to a pandas DataFrame

In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/" + splits["train"])

Edit the Dataset as following format:
<br>
data = {
    "prompts": prompts,
    "responses": responses
}


In [ ]:
def split_text(row):
    prompt = row.split("[/INST]")[0].replace("<s>[INST]", "").strip()
    response = row.split("[/INST]")[1].replace("</s>", "").strip()
    return prompt, response


df[["prompt", "response"]] = df["text"].apply(lambda x: pd.Series(split_text(x)))

prompts = df["prompt"].tolist()
responses = df["response"].tolist()

data = {
    "prompts": prompts,
    "responses": responses
}


**Select a backend**
<br>
Keras is a high-level, multi-framework deep learning API designed for simplicity and ease of use. Using Keras 3, you can run workflows on one of three backends: TensorFlow, JAX, or PyTorch. For this tutorial, configure the backend for JAX as it typically provides the better performance.

In [ ]:
import os 

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

Load Model

In [ ]:
# load model
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma_lm.summary()

The following code snippet uses a sampler object, specifically keras_hub.samplers.TopKSampler, to control how the gemma_lm.generate method selects the next token during text generation. Let's break down what sampler and TopKSampler mean in this context:

1. Sampler: In the realm of large language models (LLMs), a sampler is an algorithm or strategy that determines **which token to choose next from the model's predicted probability distribution over the vocabulary**. When an LLM generates text, it doesn't just pick the single most probable word; often, it uses sampling techniques to introduce variety and creativity, making the output less repetitive and more human-like.

2. keras_hub.samplers.TopKSampler: This is a specific type of sampling strategy. It works as follows:

3. k: The k parameter (set to 5 in your code) means that the model will consider only **the k most probable next tokens from its vocabulary**. All other tokens, even if they have a non-zero probability, are ignored for that generation step.
4. Sampling from Top-K: After identifying the top k tokens, the model then samples one token from this reduced set, typically weighted by their probabilities. This helps to balance between creativity and coherence. If k were 1, it would always pick the most probable token, leading to very deterministic and often dull text. If k were very large (or equal to the vocabulary size), it would be pure random sampling, potentially leading to nonsensical text.
seed: The seed parameter (set to 2 in your code) ensures reproducibility. If you run the generation with the same seed and other parameters, you will get the exact same output. This is useful for debugging and consistent experimentation.

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=256))

**LoRA fine-tuning**

Configure LoRA tuning
Activate LoRA tuning using the Keras model.backbone.enable_lora() method, including a LoRA rank value. The LoRA rank determines the dimensionality of the trainable matrices that are added to the original weights of the LLM. It controls the expressiveness and precision of the fine-tuning adjustments. A higher rank means more detailed changes are possible, but also means more trainable parameters. A lower rank means less computational overhead, but potentially less precise adaptation.

This example uses a LoRA rank of 4. In practice, begin with a relatively small rank (such as 4, 8, 16). This setting is computationally efficient for experimentation. Train your model with this rank and evaluate the performance improvement on your task. Gradually increase the rank in subsequent trials and see if that further boosts performance.


In [ ]:
# Enable LoRA for the model and set the LoRA rank to 5.
gemma_lm.backbone.enable_lora(rank=5)

Check the model summary after setting the LoRA rank. Notice that enabling LoRA reduces the number of trainable parameters significantly compared to the total number of parameters in the model:

In [ ]:
gemma_lm.summary()

**Run the fine-tune process**


Run the fine-tuning process using the fit() method. This process can take several minutes depending on your compute resources, data size, and number of epochs:

In [ ]:
gemma_lm.fit(data, epochs=1, batch_size=1)